# Background

The original goal of our project was to develop an AI content detector capable of identifying whether a video was AI-generated or real. As the project evolved, we refined the scope by dividing the work into two related directions: AI-generated video detection and AI-generated image detection. For Milestone 1, this notebook focuses on the image-detection branch of the project and presents a complete PyTorch training pipeline for classifying images as either real or AI-generated.


The implementation uses the EfficientNet-B0 architecture with pretrained weights. The original classification layer is replaced with a new fully connected layer for binary classification (real vs. AI-generated images). At first, only the new classifier layer is trained, while keeping the pretrained backbone frozen. After three epochs, the last two convolutional blocks are unfrozen and fine-tuned to allow the model to adapt its learned features to the AI-image detection task. Various techniques like data augmentation, dropout, and transfer learning are applied to improve generalization and reduce overfitting.



# Imports

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import kagglehub

from torchvision import datasets, models
from torch.utils.data import DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import classification_report, confusion_matrix

import numpy as np
import matplotlib.pyplot as plt

In [2]:
# move to the gpu if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device ", device)

Using device  cuda


In [3]:
# get the path to and download the CIFAKE dataset from kaggle, containing 60,000 real and 60,000 AI generated images using Stable Diffusion v4
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cifake-real-and-ai-generated-synthetic-images' dataset.
Path to dataset files: /kaggle/input/cifake-real-and-ai-generated-synthetic-images


# Data Preprocessing

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [5]:
train_dataset = datasets.ImageFolder(
    root=f"{path}/train",
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=f"{path}/test",
    transform=test_transform
)


In [6]:
# train_dataset.classes
train_dataset[1]
print(train_dataset[0][0].shape)

torch.Size([3, 224, 224])


In [7]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load Pretrained Model

In [8]:
# load EfficientNet architecture with pretrained weights on imagenet
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

# Freeze early layers
for param in model.parameters():
    param.requires_grad = False

# check how many features replace only the final layer
num_features = model.classifier[1].in_features
# add dropout and replace the final layer from num_features to 2 features (0 - real, 1 - fake)
model.classifier = nn.Sequential(
    nn.BatchNorm1d(num_features),
    nn.Dropout(0.3),
    nn.Linear(num_features, 2)
)

# train only the classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# move the model to correct hardware
model = model.to(device)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 174MB/s]


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr = 0.0001
)

# Model Training Function

In [10]:
def train(model, loader, optimizer, criterion):

    model.train()
    running_loss = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # forward pass, feed images through nn
        outputs = model(images)

        loss = criterion(outputs, labels)

        # reset gradient so they dont accumulate across batches
        optimizer.zero_grad()
        # gradient of the loss with respect to each weight
        loss.backward()
        # Adam optimizer updates the weight
        optimizer.step()

        # loss.item() converts the loss tensor into a Python number to calc avg loss later
        running_loss += loss.item()
    # return average loss per batch/epoch
    return running_loss / len(loader)

# Model Evaluation Function

In [11]:
def evaluate(model, loader):
    # switch the nn into evaluation mode i.e to disable dropout
    model.eval()

    # lists to store predicted labels and true labels
    all_preds = []
    all_labels = []

    # we are not training, so do not compute gradients
    with torch.no_grad():
        for images, labels in loader:

            # move images to the same hardware as model
            images = images.to(device)
            # forward pass, feed images through nn
            outputs = model(images)

            # find the highest value along a dimension
            _, preds = torch.max(outputs, 1)
            # move predictions back to CPU and save them in lists
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    # return true labels and predicted labels
    return all_labels, all_preds

# Training Loop

In [ ]:
epochs = 20
for epoch in range(epochs):

    # unfreeze the last two conv. blocks starting from epoch 3 (finetuning the deeper layers)
    if epoch == 3:
        for param in model.features[-1].parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-5
        )

    loss = train(model, train_loader, optimizer, criterion)

    # training metrics
    train_labels, train_preds = evaluate(model, train_loader)
    train_accuracy = np.mean(np.array(train_labels) == np.array(train_preds))

    # test metrics
    test_labels, test_preds = evaluate(model, test_loader)
    test_accuracy = np.mean(np.array(test_labels) == np.array(test_preds))

    print(f"Epoch {epoch+1}/{epochs}")
    print("Train loss:", loss)
    print("Train accuracy:", train_accuracy)
    print("Test accuracy:", test_accuracy)
    print()

Epoch 1/20
Train loss: 0.3600764737224579
Train accuracy: 0.87461
Test accuracy: 0.8415



# Model Metrics

In [ ]:
labels, preds = evaluate(model, test_loader)
print(classification_report(labels, preds))

In [ ]:
cm = confusion_matrix(labels, preds)
print(cm)

# Save Model & Load it Later

In [ ]:
torch.save(model.state_dict(), "ai_image_detector.pth")

In [ ]:
model = models.resnet50()

model.fc = nn.Linear(model.fc.in_features,2)

model.load_state_dict(torch.load("ai_fake_detector.pth"))

model.eval()

# Predict for Single Image

In [ ]:
from PIL import Image

def predict(image_path):

    image = Image.open(image_path).convert("RGB")

    image = test_transform(image)

    image = image.unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(image)

        _, pred = torch.max(outputs,1)

    return "Fake" if pred.item()==1 else "Real"